# Factorizations and sensitive measurements

[Course index](../README.md) · [Week 4](../seminars/04_lu_and_numerics.md)

**Predict → compute → explain → change an assumption.** Run top to bottom in a fresh Python kernel. GitHub previews do not execute widgets. Install the repository requirements once; no downloads occur in this notebook. All investigations have a paper route in the linked seminar sheets. A plot supports exploration, not a proof.

AI may help with the investigation if permitted. Write a prediction first, then independently verify at least one claim. Individual exit questions are completed without AI.

In [ ]:
from pathlib import Path
import sys
# Works when Jupyter starts in the repository root or in notebooks/.
root = Path.cwd() if (Path.cwd() / 'la_labs.py').exists() else Path.cwd().parent
if not (root / 'la_labs.py').exists():
    raise RuntimeError('Start Jupyter in the repository root or notebooks directory.')
if str(root) not in sys.path: sys.path.insert(0, str(root))
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import la_labs as la


## Predict first

Which work can be reused for a second right-hand side? Can a zero pivot always be repaired without a row swap? Compute one elimination step by hand before checking the factors.

In [ ]:
A = np.array([[2,1,1],[4,3,3],[2,2,3]],float)
L = np.array([[1,0,0],[2,1,0],[1,1,1]],float)
U = np.array([[2,1,1],[0,1,1],[0,0,1]],float)
assert np.allclose(L@U,A)
for b in [np.array([4,10,7.]), np.array([4,10,8.])]:
    y = la.triangular_solve(L,b,lower=True)
    x = la.triangular_solve(U,y,lower=False)
    print('b =',b,'y =',y,'x =',x,'residual =',np.linalg.norm(A@x-b))

## Sensitivity investigation

The second equation differs from the first by εy = ε + δ. Derive the exact answer on paper first. The slider `relative_change` means δ/ε, **not** relative error in the full right-hand side. Make ε smaller while keeping δ/ε fixed. A tiny residual does not guarantee closeness to the solution of the unperturbed problem. The nearly overlapping lines may look identical; read the numerical results too.

In [ ]:
def explore(log_epsilon=-4, relative_change=1):
    fig, result = la.sensitivity_figure(log_epsilon, relative_change)
    for key, value in result.items(): print(key, ':', value)
    display(fig); plt.close(fig)
sliders = la.interactive_plot(explore, {'log_epsilon': (-12,-1,1,-4), 'relative_change': (-3,3,.25,1)})

## Changed assumption: exact parameters

Use exact symbolic arithmetic to check the generic factorization, then explicitly substitute a=b=1. Does singularity destroy the factorization? Do not use the generic b/a formula at a=0.

In [ ]:
import sympy as sp
a,b = sp.symbols('a b')
M = sp.Matrix([[1,0,1],[a,a,a],[b,b,a]])
LL = sp.Matrix([[1,0,0],[a,1,0],[b,b/a,1]])
UU = sp.Matrix([[1,0,1],[0,a,0],[0,0,a-b]])
assert sp.simplify(LL@UU-M) == sp.zeros(3)
display(LL.subs({a:1,b:1}),UU.subs({a:1,b:1}))
print('Singular case determinant:',M.subs({a:1,b:1}).det())

## Explain and transfer

Classify a=0,b=0 and a=0,b≠0 using the unit-lower convention. Then explain what to do with the right-hand side under PA=LU.

**Individual exit:** why can an exact solver still produce a very different answer after a small measurement change?